# **PRE-REQUISITES AND ENVIRONMENT SETUP**


MOUNTING GOOGLE DRIVE


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
DRIVE_DATA = f"{DRIVE_BASE}/legal_pdfs"
DRIVE_CHECKPOINTS = f"{DRIVE_BASE}/checkpoints"
DRIVE_OUTPUT_CSV = f"{DRIVE_BASE}/legal_raw_data.csv"

import os
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)
print("✅ Drive mounted and folders created.")


KAGGLE API SETUP

In [ ]:
from google.colab import files
files.upload()


In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
import os
print("DRIVE_BASE =", os.path.abspath("{}".format("""/content/drive/MyDrive/rag_legal_project""")))
print("Python version and environment OK.")

In [ ]:
# Insert below DRIVE_BASE / environment cells

import os, pathlib, csv, hashlib, json

# 🔧 Base directory for the entire RAG legal project inside Google Drive
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"

# 📁 Folder where all raw PDF datasets will be stored
DATASETS_DIR = os.path.join(DRIVE_BASE, "datasets_raw")

# 📄 CSV file that stores a manifest of every file in datasets_raw (path, size, hash)
MANIFEST_PATH = os.path.join(DRIVE_BASE, "datasets_manifest.csv")

# Ensure the datasets_raw directory exists
os.makedirs(DATASETS_DIR, exist_ok=True)

# ---------------------------------------------------------------------
# 🔍 Helper function: Compute SHA-256 hash of a file
#    - Used to detect duplicates
#    - Helps verify integrity of PDFs
#    - Reads file in chunks to avoid memory overload
# ---------------------------------------------------------------------
def sha256_of_file(path, chunk_size=2**20):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)   # Read file in 1MB chunks
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# ---------------------------------------------------------------------
# 📦 Build dataset manifest
#    - Scans the entire datasets_raw directory
#    - Records:
#        • relative path
#        • absolute path
#        • file size
#        • SHA-256 hash
#    - Writes everything into datasets_manifest.csv
# ---------------------------------------------------------------------
def build_manifest(root_dir=DATASETS_DIR, out_csv=MANIFEST_PATH):
    rows = []

    # Walk through every file inside datasets_raw
    for p in pathlib.Path(root_dir).rglob("*"):
        if p.is_file():
            rows.append({
                "rel_path": str(p.relative_to(root_dir)),  # Path inside datasets_raw
                "abs_path": str(p),                        # Full absolute path
                "size_bytes": p.stat().st_size,            # File size for metadata
                "sha256": sha256_of_file(str(p))           # Unique file signature
            })

    # Write the manifest CSV
    with open(out_csv, "w", newline="", encoding="utf8") as f:
        writer = csv.DictWriter(f, fieldnames=["rel_path","abs_path","size_bytes","sha256"])
        writer.writeheader()
        writer.writerows(rows)

    print("✅ Manifest written:", out_csv, "rows=", len(rows))

# Print path to confirm directories are correctly set
print("DATASETS_DIR:", DATASETS_DIR)


In [ ]:
# run this to show top of manifest (if exists)
import pandas as pd, os
if os.path.exists(MANIFEST_PATH):
    print(pd.read_csv(MANIFEST_PATH).head())
else:
    print("Manifest not found — run build_manifest() after downloads")


In [ ]:
!pip install -q langchain chromadb pypdf2 tiktoken sentence-transformers \
                kaggle tqdm psutil faiss-cpu unstructured
# Added:
# • kaggle → access Kaggle datasets
# • tqdm → progress bars
# • psutil → check available RAM
# • faiss-cpu → optional fast vector store
# • unstructured → better PDF/text parsing fallback


In [ ]:
import importlib, sys
packages = ["langchain","chromadb","openai","pypdf","tiktoken","sentence_transformers","transformers","faiss"]
for pkg in packages:
    try:
        m = importlib.import_module(pkg)
        print(pkg, "version:", getattr(m, "__version__", "n/a"))
    except Exception as e:
        print(pkg, "import failed:", str(e))


**LOADING THE SENTENCE TRANSFORMER**

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a free, lightweight model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model_name)

print(f"Model '{model_name}' loaded successfully!")


In [ ]:
text = "The Supreme Court held that freedom of speech includes the right to express dissent."
embedding = embedding_model.encode(text)

print("Embedding vector shape:", embedding.shape)
print("First 5 dimensions:", embedding[:5])


In [ ]:
import os, shutil

# Save to Drive (only if you mounted earlier)
save_path = "/content/drive/MyDrive/legal_rag/models/all-MiniLM-L6-v2"
os.makedirs(save_path, exist_ok=True)
embedding_model.save(save_path)
print("Model saved to:", save_path)


In [ ]:
# Optional: only if you ever need to access private HuggingFace models
from getpass import getpass
import os

hf_token = getpass("Paste your Hugging Face token (optional, press Enter if not needed): ")
if hf_token:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
    print("Token set successfully!")
else:
    print("No token entered; using public models only.")


In [ ]:
!mkdir -p /content/data
!mv *.pdf /content/data/
!ls -la /content/data


In [ ]:
# 🧩 Stage 1 – Step 2: Import Core Libraries (LangChain v0.3+ compatible)
!pip install -qU langchain langchain-community langchain-core langchain-text-splitters pypdf tqdm psutil chromadb kaggle

import os, glob, json, psutil, shutil
from tqdm import tqdm
from google.colab import drive

# LangChain community + text splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("✅ Core libraries imported successfully (LangChain v0.3+).")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("🎉 TextSplitter module verified working.")


In [ ]:
!pip install -U langchain-community

# **Load all PDFs and collect metadata**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os
from tqdm import tqdm  # progress bar for visual tracking

# 📂 Folder where PDFs are stored (should contain only PDF files)
data_folder = "/content/data"

# 📝 List all PDF files in the folder (case-insensitive)
pdf_files = [f for f in os.listdir(data_folder) if f.lower().endswith(".pdf")]

all_docs = []  # Will hold every PDF page as a separate document
print(f"📚 Found {len(pdf_files)} PDF files to process...\n")

# -------------------------------------------------------------------
# 📄 Loop through each PDF file and load its pages
#    - PyPDFLoader loads each page as a Document object
#    - We attach metadata to every page to help future retrieval
# -------------------------------------------------------------------
for file in tqdm(pdf_files):
    file_path = os.path.join(data_folder, file)
    loader = PyPDFLoader(file_path)

    # Load all pages from the PDF
    docs = loader.load()

    # Add metadata to each page for traceability
    for i, d in enumerate(docs):
        d.metadata["source"] = file                # PDF file name
        d.metadata["page_number"] = i + 1          # Page index (1-based)
        d.metadata["num_pages"] = len(docs)        # Total pages in this PDF

    # Add all pages to main list
    all_docs.extend(docs)

# -------------------------------------------------------------------
# 📊 Summary output after processing all PDFs
# -------------------------------------------------------------------
print(f"\n✅ Successfully loaded {len(all_docs)} total pages across {len(pdf_files)} PDFs.")
print("Example metadata for first doc:")
print(all_docs[0].metadata)

# Show first 400 characters from the first page
print("Preview of text:\n", all_docs[0].page_content[:400])


In [ ]:
# 🧩 Stage 1 – Step 3: Setup and Verify Paths

import os, psutil

# ---------------------------------------------------------
# 📌 Base directory for your entire project inside Colab
# ---------------------------------------------------------
BASE_DIR = "/content"

# ---------------------------------------------------------
# 📁 Define key project folders:
#     - legal_pdfs:   raw PDF files
#     - chroma_db:    vector database storage
#     - checkpoints:  intermediate save/checkpoint folder
# ---------------------------------------------------------
DATA_PATH = os.path.join(BASE_DIR, "legal_pdfs")
DB_PATH = os.path.join(BASE_DIR, "chroma_db")
CHECKPOINT_PATH = os.path.join(BASE_DIR, "checkpoints")

# ---------------------------------------------------------
# 📂 Create folders if they do not exist
# ---------------------------------------------------------
for p in [DATA_PATH, DB_PATH, CHECKPOINT_PATH]:
    os.makedirs(p, exist_ok=True)

# ---------------------------------------------------------
# 🖨️ Print and verify directory structure
# ---------------------------------------------------------
print("✅ Folder structure ready:")
print(f"PDFs Folder : {DATA_PATH}")       # Where PDFs go
print(f"Vector DB   : {DB_PATH}")         # Where Chroma DB will be saved
print(f"Checkpoints : {CHECKPOINT_PATH}") # Where checkpoints are logged

# ---------------------------------------------------------
# 💾 Show available RAM (helpful for debugging memory-heavy processes)
# ---------------------------------------------------------
ram_gb = psutil.virtual_memory().total / 1e9
print(f"💾 Available RAM: {ram_gb:.2f} GB")


In [ ]:
# Step 3: Organize and save the loaded data
import pandas as pd

# Convert LangChain Document objects into structured rows
data_records = []
for d in all_docs:
    data_records.append({
        "text": d.page_content,
        "source": d.metadata.get("source", ""),
        "page_number": d.metadata.get("page_number", ""),
        "num_pages": d.metadata.get("num_pages", "")
    })

# Create a DataFrame
df = pd.DataFrame(data_records)

# Show summary
print(f"✅ Created DataFrame with {len(df)} rows (pages).")
print("📄 Columns:", list(df.columns))
print("\nSample row:")
print(df.head(2))

# Save to file
output_path = "/content/legal_raw_data.csv"
df.to_csv(output_path, index=False)
print(f"\n💾 Saved all data to: {output_path}")


In [ ]:
# 🧩 Persistent storage: Mount Google Drive and set project paths
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
DRIVE_DATA = f"{DRIVE_BASE}/legal_pdfs"
DRIVE_CHECKPOINTS = f"{DRIVE_BASE}/checkpoints"
DRIVE_OUTPUT_CSV = f"{DRIVE_BASE}/legal_raw_data.csv"
DRIVE_VALID_SUMMARY = f"{DRIVE_BASE}/valid_pdfs_summary.csv"
DRIVE_FAILED_LOG = f"{DRIVE_BASE}/failed_pdfs_log.csv"

import os
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)

print("✅ Google Drive mounted and project folders created:")
print(" - DRIVE_DATA:", DRIVE_DATA)
print(" - DRIVE_CHECKPOINTS:", DRIVE_CHECKPOINTS)
print(" - DRIVE_OUTPUT_CSV:", DRIVE_OUTPUT_CSV)


In [ ]:
!pip install -q kaggle
import os

DATA_PATH = "/content/legal_pdfs"
os.makedirs(DATA_PATH, exist_ok=True)

print("📁 PDFs will be stored at:", DATA_PATH)


# DOWNLOAD DATASETS FROM KAGGLE


In [ ]:
# ---------------------------------------------------------
# 🧩 Stage 1 – Step X: Download Kaggle datasets directly into Drive
#     This ensures the datasets persist across sessions.
# ---------------------------------------------------------

import os

# ---------------------------------------------------------
# 📁 Create the destination folder inside Google Drive
#     DRIVE_DATA must already be defined earlier.
# ---------------------------------------------------------
os.makedirs(DRIVE_DATA, exist_ok=True)

# ---------------------------------------------------------
# 📦 List of Kaggle datasets to download
#     Each dataset will be downloaded + unzipped inside DRIVE_DATA
# ---------------------------------------------------------
KAGGLE_DATASETS = [
    "vangap/indian-supreme-court-judgments",
    "nandr39/bharatiya-sakshya-adhiniyam-dataset-bsa",
    "anishparkhe0401/indian-laws",
    "keyushnisar/legal-docu"
]

print("📁 Downloading into Drive path:", DRIVE_DATA)

# ---------------------------------------------------------
# ⬇️ Loop through each dataset and download using the Kaggle CLI
#     --unzip : automatically extracts
#     --force : overwrite if exists
# ---------------------------------------------------------
for ds in KAGGLE_DATASETS:
    print(f"\n⬇️ Downloading dataset: {ds}")

    # Download + unzip into the Drive dataset folder
    os.system(f"kaggle datasets download -d {ds} -p '{DRIVE_DATA}' --unzip --force")

    # Show current folder size after each dataset download
    os.system(f"du -sh '{DRIVE_DATA}' || true")


In [ ]:
import json, os

# Save your kaggle.json correctly
kaggle_creds = {
    "username": "soundarbalajij",
    "key": "97bfcd2e3d7b52bd516f2c617e49978a"
}

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)

os.chmod("/root/.kaggle/kaggle.json", 600)

print("Kaggle authentication successful.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = "/content/drive/MyDrive/rag_legal_project/legal_pdfs"
os.makedirs(DRIVE_DATA, exist_ok=True)

print("Drive folder:", DRIVE_DATA)


In [ ]:
# 🧩 Stage 2 – Step X: Download verified PDF datasets
import os

DRIVE_DATA = "/content/drive/MyDrive/rag_legal_project/legal_pdfs"
os.makedirs(DRIVE_DATA, exist_ok=True)

PDF_DATASETS = [
    "lawrencedhakal/indian-bare-acts-pdf",
    "lawrencedhakal/indian-laws-and-rules-dataset",
    "lci-india/law-commission-of-india-reports",
    "openlegaldata/india-legal-case-pdfs",
    "legaltechindia/indian-court-judgment-pdfs",
    "priyanshukumar2020/indian-police-act-bare-act-pdf",
    "justicehub/indian-judgments-pdf-dataset"
]

print("📁 Downloading legal PDF datasets into:", DRIVE_DATA)

for ds in PDF_DATASETS:
    print(f"\n⬇️ Downloading dataset: {ds}")
    os.system(f"kaggle datasets download -d {ds} -p '{DRIVE_DATA}' --unzip --force")

print("\n🎉 All PDF datasets downloaded.")


In [ ]:
!ls -R "/content/drive/MyDrive/rag_legal_project/legal_pdfs"

In [ ]:
!find "/content/drive/MyDrive/rag_legal_project/legal_pdfs" -type f -iname "*.pdf" | wc -l


In [ ]:
!find "/content/drive/MyDrive/rag_legal_project/legal_pdfs" -maxdepth 5 -type f | wc -l


In [ ]:
!find "/content/drive/MyDrive/rag_legal_project/legal_pdfs" -type f | sed 's/.*\.//' | sort | uniq -c


# **PDF** VALIDATION TO ENSURE READABILITY


In [ ]:
# 🧩 Stage 2 – Step 2: Validate PDFs (resume-aware, writes to Drive)
# Purpose: Checks each PDF in Drive for readability before text extraction.
# Automatically skips already validated files and saves results back to Drive.

from langchain_community.document_loaders import PyPDFLoader
import pandas as pd
from tqdm import tqdm
import os, math

# --- ✅ Ensure Drive paths are defined ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
DRIVE_DATA = f"{DRIVE_BASE}/legal_pdfs"
DRIVE_VALID_SUMMARY = f"{DRIVE_BASE}/valid_pdfs_summary.csv"
DRIVE_FAILED_LOG = f"{DRIVE_BASE}/failed_pdfs_log.csv"

PDF_DIR = DRIVE_DATA  # Folder containing your PDFs
valid_docs = []
failed_files = []

# --- Step 1: Collect PDFs ---
all_pdfs = [
    os.path.join(root, f)
    for root, _, files in os.walk(PDF_DIR)
    for f in files if f.lower().endswith(".pdf")
]
print(f"🔍 Found {len(all_pdfs)} total PDF files in {PDF_DIR}")

# --- Step 2: Resume if previous summary exists ---
if os.path.exists(DRIVE_VALID_SUMMARY):
    try:
        prev = pd.read_csv(DRIVE_VALID_SUMMARY)
        processed_seen = set(prev['file'].tolist())
        print(f"▶ Resuming: {len(processed_seen)} previously validated files found in {DRIVE_VALID_SUMMARY}")
    except Exception:
        processed_seen = set()
else:
    processed_seen = set()

# --- Step 3: Quick sanity check on first few PDFs ---
print("\n🧪 Quick test of first 3 files (if any)")
for f in all_pdfs[:3]:
    print(" -", f)
    try:
        loader = PyPDFLoader(f)
        pages = loader.load()
        print(f"   ✅ {os.path.basename(f)} -> {len(pages)} pages")
    except Exception as e:
        print(f"   ❌ {os.path.basename(f)} -> {e}")

# --- Step 4: Full validation loop (skips already processed) ---
for f in tqdm(all_pdfs, desc="Validating PDFs"):
    if os.path.basename(f) in processed_seen or f in processed_seen:
        continue
    try:
        loader = PyPDFLoader(f)
        pages = loader.load()
        if len(pages) == 0:
            raise ValueError("No pages found")
        valid_docs.append({"file": f, "pages": len(pages)})
    except Exception as e:
        failed_files.append({"file": f, "error": str(e)})

# --- Step 5: Merge results + Save back to Drive ---
if os.path.exists(DRIVE_VALID_SUMMARY):
    prev_df = pd.read_csv(DRIVE_VALID_SUMMARY)
    merged = pd.concat([prev_df, pd.DataFrame(valid_docs)], ignore_index=True)
    merged = merged.drop_duplicates(subset=['file'])
else:
    merged = pd.DataFrame(valid_docs)

merged.to_csv(DRIVE_VALID_SUMMARY, index=False)
pd.DataFrame(failed_files).to_csv(DRIVE_FAILED_LOG, index=False)

# --- Step 6: Summary Output ---
print(f"\n✅ Validation saved to {DRIVE_VALID_SUMMARY} (valid: {len(merged)})")
print(f"⚠️ Failed list saved to {DRIVE_FAILED_LOG} (failed: {len(failed_files)})")


In [ ]:
# ----------------------------------------------
# 📌 Stage 2 – Step 2: PDF Validation (Final)
#     Validates ONLY new PDFs inside the 3 folders:
#     - law_commission
#     - bare_acts
#     - constitution
# ----------------------------------------------

!pip install pypdf langdetect tqdm

import os
import warnings
from pypdf import PdfReader
from langdetect import detect
from tqdm import tqdm
import pandas as pd
from google.colab import drive

# -------------------------
# 🔗 Mount Google Drive
#     Required because PDFs live inside Drive
# -------------------------
drive.mount('/content/drive')

# -------------------------
# 📂 Base folder containing 3 legal PDF subfolders
# -------------------------
BASE_DIR = "/content/drive/MyDrive/rag_legal_project/legal_pdfs"

# Subfolders that will be scanned + validated
TARGET_FOLDERS = [
    os.path.join(BASE_DIR, "law_commission"),
    os.path.join(BASE_DIR, "bare_acts"),
    os.path.join(BASE_DIR, "constitution"),
]

# CSVs that store validation results persistently
VALID_CSV = f"{BASE_DIR}/validated_pdfs.csv"
FAILED_CSV = f"{BASE_DIR}/failed_pdfs.csv"

# Ensure base folder is present
os.makedirs(BASE_DIR, exist_ok=True)

# -------------------------
# 🧩 Load Previous Results
#     This ensures we DO NOT re-validate old files.
# -------------------------
if os.path.exists(VALID_CSV):
    old_valid = pd.read_csv(VALID_CSV)["path"].tolist()
else:
    old_valid = []

if os.path.exists(FAILED_CSV):
    old_failed = pd.read_csv(FAILED_CSV)["path"].tolist()
else:
    old_failed = []

print(f"▶ Found {len(old_valid)} previously validated PDFs — skipping them")
print(f"▶ Found {len(old_failed)} previously failed PDFs — skipping them")

# -------------------------
# 📥 Collect ONLY NEW PDFs
#     We walk inside all 3 subfolders and collect only
#     PDFs that do NOT exist in the old CSVs.
# -------------------------
new_pdfs = []

for folder in TARGET_FOLDERS:
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".pdf"):
                full_path = os.path.join(root, f)

                # Only add if NOT validated before
                if full_path not in old_valid and full_path not in old_failed:
                    new_pdfs.append(full_path)

print(f"\n🔍 NEW PDFs to validate: {len(new_pdfs)}\n")

# -------------------------
# 🧪 PDF Validation Function
#     Rules:
#       - PDF must open successfully
#       - First 2 pages must contain extractable text
#       - First 500 characters must be detected as English
# -------------------------
def is_valid_english_pdf(path):
    try:
        reader = PdfReader(path)

        # Extract text from first 2 pages only
        text = ""
        for page in reader.pages[:2]:
            text += page.extract_text() or ""

        # Reject if no text at all
        if not text.strip():
            return False

        # Detect language only from first 500 chars
        lang = detect(text[:500])
        return lang == "en"

    except Exception:
        # Any error = invalid PDF
        return False

# -------------------------
# 🚀 Validate ONLY New PDFs
#     Progress bar shows real-time validation status
# -------------------------
valid_list = []
failed_list = []

print("Validating new PDFs:")

for pdf in tqdm(new_pdfs):
    if is_valid_english_pdf(pdf):
        valid_list.append(pdf)
    else:
        failed_list.append(pdf)

# -------------------------
# 💾 Save Newly Generated Results
#     Results are appended to existing CSVs
#     to maintain a persistent record.
# -------------------------
if valid_list:
    df_val = pd.DataFrame({"path": valid_list})
    if os.path.exists(VALID_CSV):
        df_val.to_csv(VALID_CSV, mode="a", header=False, index=False)
    else:
        df_val.to_csv(VALID_CSV, index=False)

if failed_list:
    df_fail = pd.DataFrame({"path": failed_list})
    if os.path.exists(FAILED_CSV):
        df_fail.to_csv(FAILED_CSV, mode="a", header=False, index=False)
    else:
        df_fail.to_csv(FAILED_CSV, index=False)

# -------------------------
# 🎉 Summary Output
# -------------------------
print("\n🎉 PDF Validation Completed!")
print(f"✔ Valid PDFs added: {len(valid_list)}")
print(f"✘ Failed PDFs added: {len(failed_list)}")


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/rag_legal_project/legal_pdfs"   # change if your base path is different

folders = ["constitution", "bare_acts", "law_commission"]

for folder in folders:
    path = os.path.join(BASE_DIR, folder)
    if not os.path.exists(path):
        print(f"❌ Folder not found: {folder}")
        continue

    pdf_count = len([f for f in os.listdir(path) if f.lower().endswith(".pdf")])
    print(f"📄 {folder}: {pdf_count} PDF files")


# **EXTRACT TEXT FROM VALIDATED PDFS FROM PYPDFLOADER**

In [ ]:
# 🧩 Stage 2 – Step 3: Extract text from validated PDFs → Drive-backed CSV (resumable)
# Paste this cell directly BELOW your Stage 2 Step 2 (validation) cell.

from langchain_community.document_loaders import PyPDFLoader
import pandas as pd, json, os, time
from tqdm import tqdm

# Paths (ensure these match your Drive mount)
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
DRIVE_VALID_SUMMARY = os.path.join(DRIVE_BASE, "valid_pdfs_summary.csv")
DRIVE_OUTPUT_CSV = os.path.join(DRIVE_BASE, "legal_raw_data.csv")
DRIVE_CHECKPOINTS = os.path.join(DRIVE_BASE, "checkpoints")
os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)

# Config — tune these if needed
BATCH_FILES = 500        # process this many PDF files before saving a checkpoint
TEST_MODE = False        # True => quick smoke test
TEST_N = 5               # only used when TEST_MODE True

# Load validated list
if not os.path.exists(DRIVE_VALID_SUMMARY):
    raise FileNotFoundError("Validated list not found. Run Stage 2 Step 2 first.")
valid_df = pd.read_csv(DRIVE_VALID_SUMMARY)
valid_files = list(valid_df['file'].tolist())

if TEST_MODE:
    valid_files = valid_files[:TEST_N]
    print("⚠️ TEST_MODE on — processing", len(valid_files), "files only")

# Determine already processed sources to skip (resume logic)
processed_sources = set()
if os.path.exists(DRIVE_OUTPUT_CSV):
    try:
        existing = pd.read_csv(DRIVE_OUTPUT_CSV, usecols=["source"], dtype=str)
        processed_sources = set(existing["source"].unique())
        print(f"▶ Resuming: {len(processed_sources)} sources already processed.")
    except Exception:
        print("▶ Could not read existing output header; will append new rows.")

# Helpers
def save_checkpoint(batch_id, rows):
    path = os.path.join(DRIVE_CHECKPOINTS, f"checkpoint_{batch_id}.ndjson")
    with open(path, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("⚑ checkpoint saved:", path)

def append_rows(rows):
    if not rows:
        return
    df = pd.DataFrame(rows)
    if not os.path.exists(DRIVE_OUTPUT_CSV):
        df.to_csv(DRIVE_OUTPUT_CSV, index=False)
    else:
        df.to_csv(DRIVE_OUTPUT_CSV, mode="a", header=False, index=False)

# Main loop
batch_rows = []
batch_id = 0
processed_count = 0
start = time.time()

for fpath in tqdm(valid_files, desc="Extracting validated PDFs"):
    fname = os.path.basename(fpath)
    if fname in processed_sources or fpath in processed_sources:
        processed_count += 1
        continue
    try:
        loader = PyPDFLoader(fpath)
        pages = loader.load()
        num_pages = len(pages)
        for i, doc in enumerate(pages, start=1):
            text = doc.page_content.strip()
            if not text:
                continue
            row = {
                "text": text,
                "source": fname,
                "source_path": fpath,
                "page_number": i,
                "num_pages": num_pages
            }
            batch_rows.append(row)
    except Exception as e:
        # keep an errors file in checkpoints
        with open(os.path.join(DRIVE_CHECKPOINTS, "extract_errors.log"), "a", encoding="utf-8") as eh:
            eh.write(f"{fpath}\t{repr(e)}\n")

    processed_count += 1

    # flush regularly
    if processed_count % BATCH_FILES == 0 and batch_rows:
        append_rows(batch_rows)
        batch_id += 1
        save_checkpoint(batch_id, batch_rows)
        batch_rows = []

# final flush
if batch_rows:
    append_rows(batch_rows)
    batch_id += 1
    save_checkpoint(batch_id, batch_rows)
    batch_rows = []

elapsed = (time.time() - start) / 60.0
print(f"\n✅ Extraction complete. Processed {processed_count} files in {elapsed:.2f} minutes.")
print("➡ Output CSV:", DRIVE_OUTPUT_CSV)
print("➡ Checkpoints:", DRIVE_CHECKPOINTS)

# quick sanity check (first few rows)
if os.path.exists(DRIVE_OUTPUT_CSV):
    display(pd.read_csv(DRIVE_OUTPUT_CSV, nrows=5))
    total_rows = sum(1 for _ in open(DRIVE_OUTPUT_CSV, "r", encoding="utf-8")) - 1
    print("Total rows (pages) in CSV:", total_rows)
else:
    print("No output CSV produced — check extract_errors.log in checkpoints.")


In [ ]:
# ----------------------------------------------
# 🧩 Stage 2 – Step 3: Extract text from VALIDATED PDFs → Drive-backed CSV
# ----------------------------------------------

!pip install langchain_community pypdf

from langchain_community.document_loaders import PyPDFLoader
import pandas as pd, json, os, time
from tqdm import tqdm

# Base paths
BASE_DIR = "/content/drive/MyDrive/rag_legal_project/legal_pdfs"
VALIDATED_CSV = f"{BASE_DIR}/validated_pdfs.csv"   # from Step 2
OUTPUT_CSV = "/content/drive/MyDrive/rag_legal_project/legal_raw_data.csv"
CHECKPOINT_DIR = "/content/drive/MyDrive/rag_legal_project/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# -------------------------
# Load validated PDFs
# -------------------------
if not os.path.exists(VALIDATED_CSV):
    raise FileNotFoundError("validated_pdfs.csv not found — run Stage 2 Step 2 first.")

validated_df = pd.read_csv(VALIDATED_CSV)
validated_files = validated_df["path"].tolist()

print(f"📄 Validated PDFs loaded: {len(validated_files)}")

# -------------------------
# Resume logic — skip processed PDFs
# -------------------------
processed_sources = set()
if os.path.exists(OUTPUT_CSV):
    try:
        existing = pd.read_csv(OUTPUT_CSV, usecols=["source_path"])
        processed_sources = set(existing["source_path"].unique())
        print(f"🔁 Resuming — already processed: {len(processed_sources)} PDFs")
    except:
        print("⚠️ Could not read existing CSV fully, will append new rows.")

# -------------------------
# Helper: save batch as checkpoint
# -------------------------
def save_checkpoint(batch_id, rows):
    cp_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_{batch_id}.ndjson")
    with open(cp_path, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"💾 Checkpoint saved: {cp_path}")

def append_rows(rows):
    if not rows:
        return
    df = pd.DataFrame(rows)
    if not os.path.exists(OUTPUT_CSV):
        df.to_csv(OUTPUT_CSV, index=False)
    else:
        df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)

# -------------------------
# Extraction loop
# -------------------------
batch_rows = []
batch_id = 0
processed = 0
BATCH_SIZE = 300  # Tune based on speed

print("\n🚀 Starting extraction...\n")

for fpath in tqdm(validated_files, desc="Extracting PDFs"):
    if fpath in processed_sources:
        continue

    fname = os.path.basename(fpath)

    try:
        loader = PyPDFLoader(fpath)
        pages = loader.load()
        num_pages = len(pages)

        for i, doc in enumerate(pages, start=1):
            text = doc.page_content.strip()
            if not text:
                continue
            batch_rows.append({
                "text": text,
                "source": fname,
                "source_path": fpath,
                "page_number": i,
                "num_pages": num_pages
            })

    except Exception as e:
        with open(os.path.join(CHECKPOINT_DIR, "extract_errors.log"), "a", encoding="utf-8") as log:
            log.write(f"{fpath}\t{repr(e)}\n")

    processed += 1

    if processed % BATCH_SIZE == 0 and batch_rows:
        append_rows(batch_rows)
        batch_id += 1
        save_checkpoint(batch_id, batch_rows)
        batch_rows = []

# Flush last batch
if batch_rows:
    append_rows(batch_rows)
    batch_id += 1
    save_checkpoint(batch_id, batch_rows)
    batch_rows = []

print("\n🎉 Extraction completed!")
print("➡ Output CSV:", OUTPUT_CSV)
print("➡ Checkpoints:", CHECKPOINT_DIR)

# Show a sanity check
if os.path.exists(OUTPUT_CSV):
    display(pd.read_csv(OUTPUT_CSV, nrows=5))


In [ ]:
import json, os, pandas as pd

checkpoint_dir = "/content/drive/MyDrive/rag_legal_project/checkpoints"
rows = []
for f in os.listdir(checkpoint_dir):
    if f.endswith(".ndjson"):
        with open(os.path.join(checkpoint_dir, f), encoding="utf-8") as fh:
            rows.extend(json.loads(line) for line in fh)
merged = pd.DataFrame(rows)
merged.to_csv("/content/drive/MyDrive/rag_legal_project/legal_raw_data.csv", index=False)
print("✅ Merged checkpoints into single CSV with", len(merged), "rows")


# **CHUNKING THE TEXTS...**

In [ ]:
# 🧩 Stage 3 – Step 2: Text Chunking for Embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
import pandas as pd, os
from tqdm import tqdm

DRIVE_CHUNKS = f"{DRIVE_BASE}/chunks.parquet"

# --- Load data ---
df = pd.read_csv(f"{DRIVE_BASE}/legal_raw_data.csv")
print("📖 Processing", len(df), "rows")

# --- Define text splitter ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # each chunk ~1000 characters
    chunk_overlap=150,  # small overlap for context continuity
    separators=["\n\n", "\n", ".", " ", ""]
)

chunk_rows = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    chunks = splitter.split_text(str(row["text"]))
    for idx, chunk in enumerate(chunks):
        chunk_rows.append({
            "source": row["source"],
            "page_number": row["page_number"],
            "chunk_id": f"{row['source']}_p{row['page_number']}_c{idx}",
            "text_chunk": chunk
        })

chunks_df = pd.DataFrame(chunk_rows)
chunks_df.to_parquet(DRIVE_CHUNKS, index=False)
print(f"✅ Saved {len(chunks_df):,} chunks to {DRIVE_CHUNKS}")


In [ ]:
# -------------------------------------------------------------
# 🧩 Stage 3 – Step 2: Text Chunking ONLY for Subfolder PDFs
#
# This stage reads the CSV containing extracted text from
# validated subfolder PDFs (law_commission, bare_acts,
# constitution), splits each page into smaller chunks, and
# stores them in a separate parquet file.
#
# IMPORTANT:
#   - This does NOT touch or overwrite your OLD chunk files.
#   - This produces a NEW file: chunks_subfolders.parquet
# -------------------------------------------------------------

from langchain.text_splitter import RecursiveCharacterTextSplitter
import pandas as pd, os
from tqdm import tqdm

# Base project directory inside Google Drive
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"

# 🔹 Output path for NEW chunk file (subfolders ONLY)
DRIVE_CHUNKS_SUBFOLDERS = f"{DRIVE_BASE}/chunks_subfolders.parquet"

# -------------------------------------------------------------
# 📥 Load ONLY text extracted from validated subfolder PDFs
#     This CSV was generated by the earlier extraction script.
#     It contains:
#       - source (filename)
#       - source_path
#       - page_number
#       - text (raw extracted page text)
# -------------------------------------------------------------
RAW_DATA = f"{DRIVE_BASE}/legal_raw_data.csv"
df = pd.read_csv(RAW_DATA)

print("📖 Processing", len(df), "rows from validated subfolder PDFs")

# -------------------------------------------------------------
# 🔪 Define text splitter
#     Recursive splitter ensures clean chunk boundaries:
#       - ~1000 characters per chunk
#       - 150 characters overlap to preserve context
#       - Progressive fallback separators
# -------------------------------------------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

# -------------------------------------------------------------
# 🧩 Generate chunks for each page
#     For each row in the CSV:
#       - Split its text into multiple chunks
#       - Assign metadata (source, page, chunk_id, etc.)
# -------------------------------------------------------------
chunk_rows = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    # Split the page text into multiple chunks
    chunks = splitter.split_text(str(row["text"]))

    # Store each chunk as an entry with metadata
    for idx, chunk in enumerate(chunks):
        chunk_rows.append({
            "source": row["source"],                  # filename
            "source_path": row["source_path"],        # full path
            "page_number": row["page_number"],        # page number
            "chunk_id": f"{row['source']}_p{row['page_number']}_c{idx}",
            "text_chunk": chunk                       # actual text chunk
        })

# -------------------------------------------------------------
# 💾 Save all chunks to parquet
#     This is your dedicated subfolder-only chunk database.
# -------------------------------------------------------------
chunks_df = pd.DataFrame(chunk_rows)
chunks_df.to_parquet(DRIVE_CHUNKS_SUBFOLDERS, index=False)

print(f"✅ Completed Text Chunking for Validated Subfolder PDFs")
print(f"➡ Total Chunks Created: {len(chunks_df):,}")
print(f"➡ Saved To: {DRIVE_CHUNKS_SUBFOLDERS}")


In [ ]:
# 🧩 Stage 3 – Step 3: Generate (Resumable) Embeddings and Save to Drive
from sentence_transformers import SentenceTransformer
import pandas as pd, numpy as np, os
from tqdm import tqdm

DRIVE_EMB = f"{DRIVE_BASE}/embeddings.parquet"
DRIVE_CHUNKS = f"{DRIVE_BASE}/chunks.parquet"

# --- Load chunks ---
chunks_df = pd.read_parquet(DRIVE_CHUNKS)
print("📦 Loaded", len(chunks_df), "chunks for embedding")

# --- Check for existing embeddings ---
if os.path.exists(DRIVE_EMB):
    existing = pd.read_parquet(DRIVE_EMB)
    done = set(existing["text_chunk"].tolist())
    print(f"▶ Resuming: {len(done)} chunks already embedded.")
else:
    existing = pd.DataFrame()
    done = set()

# --- Load embedding model ---
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
model.max_seq_length = 512

# --- Encode remaining chunks ---
new_rows = []
for _, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Embedding Chunks"):
    text = row["text_chunk"]
    if text in done:
        continue
    emb = model.encode(text, show_progress_bar=False, convert_to_numpy=True)
    new_rows.append({
        "text_chunk": text,
        "source": row["source"],
        "embedding": emb
    })

# --- Merge + Save ---
if len(new_rows) > 0:
    new_df = pd.DataFrame(new_rows)
    final_df = pd.concat([existing, new_df], ignore_index=True)
    final_df.to_parquet(DRIVE_EMB, index=False)
    print(f"✅ Added {len(new_rows)} new embeddings (total {len(final_df)})")
else:
    print("✅ All chunks already processed.")
print("📁 Saved to:", DRIVE_EMB)


VERIFYING THE EMBEDDINGS

In [ ]:
# 🧩 Stage 3 – Step 4: Verify Embedding File
import pandas as pd, os

DRIVE_EMB = f"{DRIVE_BASE}/embeddings.parquet"

if os.path.exists(DRIVE_EMB):
    df = pd.read_parquet(DRIVE_EMB)
    print("✅ Embedding file found with", len(df), "rows")
    print("\n🔢 Embedding vector length:", len(df['embedding'][0]))
    print(df.head(2))
else:
    print("❌ embeddings.parquet not found — rerun Stage 3 Step 3.")


In [ ]:
# 🔍 Quick Verification
df = pd.read_parquet(f"{DRIVE_BASE}/embeddings.parquet")
print(df.head(2))
print("\n🔢 Embedding vector length:", len(df['embedding'][0]))


In [ ]:
!pip install -qU faiss-cpu langchain-huggingface langchain-community


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import pickle, os


In [ ]:
import pandas as pd, glob, os
from tqdm import tqdm

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
parquet_files = sorted(glob.glob(f"{DRIVE_BASE}/embeddings_batch_*.parquet"))

print(f"🧩 Found {len(parquet_files)} batches to combine for FAISS index.")

# --- Combine all batches ---
texts, metadatas = [], []
for path in tqdm(parquet_files):
    df = pd.read_parquet(path)
    texts.extend(df["text_chunk"].tolist())
    metadatas.extend(df[["source", "page_number"]].to_dict(orient="records"))

print(f"✅ Total text chunks loaded: {len(texts)}")


# **FAISS INITIALIZATION AND BATCH-WISE PROCESSING**

In [ ]:
# 🧩 Stage 5 – FAISS Vector Store (Batch 1 Initialization with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os, numpy as np, time

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
os.makedirs(FAISS_PATH, exist_ok=True)

# --- Load the first batch ---
batch_file = f"{DRIVE_BASE}/embeddings_batch_1.parquet"
df = pd.read_parquet(batch_file)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 1 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Progressive embedding creation ---
print("🚀 Building FAISS vector store with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(texts=batch_texts, embedding=embedding_function, metadatas=batch_metas)
    partial_stores.append(batch_store)

print("🧩 Merging all mini-batches into one FAISS index...")
vectorstore = partial_stores[0]
for ps in tqdm(partial_stores[1:], desc="📦 Merging", unit="batch"):
    vectorstore.merge_from(ps)

vectorstore.save_local(FAISS_PATH)
print(f"✅ FAISS index built & saved successfully → {FAISS_PATH}")
print(f"📊 Total vectors stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5B – FAISS Vector Store (Batch 2 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_2.parquet"

# --- Load batch 2 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 2 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches for batch 2 ---
print("🚀 Embedding new Batch 2 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge new mini-batches into the existing FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 2", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save the updated FAISS index ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 2 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5C – FAISS Vector Store (Batch 3 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_3.parquet"   # ← CHANGE HERE ONLY

# --- Load batch 3 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 3 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches for batch 3 ---
print("🚀 Embedding new Batch 3 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge batch 3 embeddings ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 3", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 3 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5D – FAISS Vector Store (Batch 4 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_4.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 4 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 4 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches for batch 4 ---
print("🚀 Embedding new Batch 4 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge batch 4 into main FAISS ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 4", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 4 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5E – FAISS Vector Store (Batch 5 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_5.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 5 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 5 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches for batch 5 ---
print("🚀 Embedding new Batch 5 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge batch 5 into existing FAISS ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 5", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 5 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5F – FAISS Vector Store (Batch 6 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_6.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 6 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 6 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches for batch 6 ---
print("🚀 Embedding new Batch 6 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge batch 6 into existing FAISS ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 6", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 6 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5G – FAISS Vector Store (Batch 7 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_7.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 7 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 7 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches for batch 7 ---
print("🚀 Embedding new Batch 7 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge batch 7 into existing FAISS ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 7", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 7 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5H – FAISS Vector Store (Batch 8 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_8.parquet"   # ← CHANGE ONLY THIS

# --- Load batch 8 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 8 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches for batch 8 ---
print("🚀 Embedding new Batch 8 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge batch 8 into existing FAISS ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 8", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 8 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5I – FAISS Vector Store (Batch 9 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_9.parquet"   # ← CHANGE ONLY THIS

# --- Load batch 9 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 9 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches for batch 9 ---
print("🚀 Embedding new Batch 9 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge batch 9 embeddings ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 9", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 9 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5J – FAISS Vector Store (Batch 10 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_10.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 10 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 10 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 10 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge vectors ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 10", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save final FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 10 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5K – FAISS Vector Store (Batch 11 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_11.parquet"   # ← ONLY EDITED HERE

# --- Load batch 11 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 11 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 11 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge vectors ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 11", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save final FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 11 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5L – FAISS Vector Store (Batch 12 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_12.parquet"   # ← ONLY EDITED HERE

# --- Load batch 12 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 12 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 12 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge vectors ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 12", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 12 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5M – FAISS Vector Store (Batch 13 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_13.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 13 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 13 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 13 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge into FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 13", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 13 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5N – FAISS Vector Store (Batch 14 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_14.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 14 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 14 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 14 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge into FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 14", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 14 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5O – FAISS Vector Store (Batch 15 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_15.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 15 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 15 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 15 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge into FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 15", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 15 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5P – FAISS Vector Store (Batch 16 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_16.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 16 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 16 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 16 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge into FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 16", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 16 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5Q – FAISS Vector Store (Batch 17 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_17.parquet"   # ← CHANGE HERE ONLY

# --- Load batch 17 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 17 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 17 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge into FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 17", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 17 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5R – FAISS Vector Store (Batch 18 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_18.parquet"   # ← CHANGE HERE ONLY

# --- Load batch 18 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 18 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 18 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge into FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 18", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 18 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5S – FAISS Vector Store (Batch 19 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_19.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 19 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 19 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 19 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge into FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 19", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 19 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

# 🧩 Stage 5S – FAISS Vector Store (Batch 20 Update with Progress Tracking)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd, os

# --- Paths ---
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"
BATCH_FILE = f"{DRIVE_BASE}/embeddings_batch_20.parquet"   # ← ONLY CHANGE HERE

# --- Load batch 20 data ---
df = pd.read_parquet(BATCH_FILE)
texts = df["text_chunk"].tolist()
metadatas = df[["source", "page_number"]].to_dict(orient="records")

print(f"📘 Loaded batch 20 with {len(texts):,} text chunks")

# --- Initialize embedding model ---
embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# --- Load existing FAISS index ---
print("🔄 Loading existing FAISS index from Drive...")
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)
print(f"📦 Existing FAISS index loaded. Current vectors: {len(vectorstore.index_to_docstore_id):,}")

# --- Build new mini-batches ---
print("🚀 Embedding new Batch 20 data with progress tracking...")
BATCH_SIZE = 500
partial_stores = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="🔄 Embedding new batches", unit="batch"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metas = metadatas[i:i + BATCH_SIZE]
    batch_store = FAISS.from_texts(
        texts=batch_texts,
        embedding=embedding_function,
        metadatas=batch_metas
    )
    partial_stores.append(batch_store)

# --- Merge into FAISS index ---
print("🧩 Merging new mini-batches into the main FAISS index...")
for ps in tqdm(partial_stores, desc="📦 Merging batch 20", unit="batch"):
    vectorstore.merge_from(ps)

# --- Save updated FAISS ---
vectorstore.save_local(FAISS_PATH)
print(f"✅ Batch 20 vectors added successfully and saved → {FAISS_PATH}")
print(f"📊 Total vectors now stored: {len(vectorstore.index_to_docstore_id):,}")


In [ ]:
import glob, pandas as pd, os

DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
parquet_files = sorted(glob.glob(f"{DRIVE_BASE}/embeddings_batch_*.parquet"))
print("📦 Found", len(parquet_files), "parquet batch files")

expected = 0
for p in parquet_files:
    # read only the length to be fast
    df = pd.read_parquet(p, columns=[])   # many versions support reading just metadata; if not, fallback below
    # if above errors, use: df = pd.read_parquet(p); expected += len(df)
    try:
        nrows = int(pd.read_parquet(p, engine="pyarrow").shape[0])
    except Exception:
        nrows = len(pd.read_parquet(p))
    expected += nrows

print("📊 Expected total vectors (sum of rows across all batches):", expected)


In [ ]:
!pip install -q langchain_community langchain_huggingface faiss-cpu

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
DRIVE_BASE = "/content/drive/MyDrive/rag_legal_project"
FAISS_PATH = f"{DRIVE_BASE}/faiss_index"

embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

print("🔄 Loading FAISS index from:", FAISS_PATH)
vectorstore = FAISS.load_local(FAISS_PATH, embeddings=embedding_function, allow_dangerous_deserialization=True)

# two ways to get size depending on langchain version:
size = None
try:
    size = len(vectorstore.index_to_docstore_id)
except Exception:
    try:
        size = vectorstore.index.ntotal
    except Exception:
        # fallback - count docstore ids
        size = len(list(vectorstore.docstore._dict.keys()))

print("📦 Actual vectors currently in FAISS:", size)


In [ ]:
# peek 5 random IDs and print metadata
import random
ids = list(vectorstore.index_to_docstore_id.keys()) if hasattr(vectorstore, "index_to_docstore_id") else list(vectorstore.docstore._dict.keys())
sample = random.sample(ids, min(5, len(ids)))

print("🔎 Sample doc IDs and metadata:")
for sid in sample:
    try:
        meta = vectorstore.docstore[sid]  # returns Document or metadata depending on implementation
    except Exception:
        meta = vectorstore.docstore._dict.get(sid)
    print(sid, "->", meta)


# SIMPLE **RETRIEVER**

In [ ]:
# Install required packages
!pip install langchain langchain-community langchain-huggingface faiss-cpu

# --- Create simple retriever and test queries ---
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
embedding_function = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

FAISS_PATH = "/content/drive/MyDrive/rag_legal_project/faiss_index"

# Load vector store
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings=embedding_function,
    allow_dangerous_deserialization=True
)

# Build retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test queries
test_queries = [
    "What weapon did Mahesh use to attack Krishna Kumar?",
    "Why did the Supreme Court apply Exception 4 to Section 300 IPC in this case?",
    "who drafted the indian constitution and when india got independence"
]

for q in test_queries:
    print("\n🔍 Query:", q)

    docs = (
        retriever.get_relevant_documents(q)
        if hasattr(retriever, "get_relevant_documents")
        else retriever.invoke(q)
    )

    if not docs:
        print("⚠️ No docs returned!")
        continue

    for i, d in enumerate(docs, start=1):
        meta = getattr(d, "metadata", getattr(d, "meta", {}))
        content = getattr(d, "page_content", getattr(d, "content", str(d)))

        print(f"\nResult {i}: Source:{meta.get('source','N/A')} Page:{meta.get('page_number','N/A')}")
        print(content[:400].replace("\n", " ") + "...")


In [ ]:
# =========================
# ALL-IN-ONE RAG TEST CELL
# Model for testing: o1-mini (smallest)
# =========================

!pip install -qU "langchain-community>=0.3.0" langchain-huggingface faiss-cpu openai transformers

import os
import pprint
from openai import OpenAI
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# 🔑 Set your key correctly here
os.environ[""] = ""  # << paste key here

client = OpenAI(api_key=os.environ[""])

# 📌 FAISS path
FAISS_PATH = "/content/drive/MyDrive/rag_legal_project/faiss_index"

print("🔄 Loading FAISS index and embeddings...")
embedding_fn = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = FAISS.load_local(FAISS_PATH, embedding_fn, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ Retriever Ready")

# 🔹 Chat function using cheapest model
def ask_openai_chat(prompt):
    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are a legal assistant. Use provided context only."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=300
    )
    return resp.choices[0].message.content

# 🔹 RAG pipeline
def rag_answer(query, show_docs=True):
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant docs found."

    if show_docs:
        print("\n--- Retrieved Docs Preview ---")
        for i, d in enumerate(docs, 1):
            print(f"\n{i})", d.page_content[:400], "...")

    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"""
Use ONLY this context to answer:

{context}

Question: {query}

Answer briefly and cite doc numbers.
"""

    return ask_openai_chat(prompt)

# 🧪 Test Query
query = "Why did the Supreme Court apply Exception 4 to Section 300 IPC?"
print("\n🔍 Query:", query)
print("\n🟦 RAG Answer:\n")
print(rag_answer(query))


In [ ]:
# --- Cell: Install compatible packages (run once) ---
# Warning: uninstalling system packages can be destructive in some environments.
# Remove the `!pip uninstall` block if you don't want to force reinstall.
!pip uninstall -y torch transformers peft accelerate trl datasets pandas sentencepiece huggingface_hub tqdm bitsandbytes triton || true

!pip install --upgrade --force-reinstall \
  torch==2.3.0 \
  transformers==4.42.4 \
  peft==0.11.1 \
  accelerate==0.33.0 \
  trl==0.9.6 \
  datasets==2.20.0 \
  pandas==2.2.2 \
  sentencepiece==0.2.0 \
  huggingface_hub==0.23.4 \
  tqdm \
  safetensors==0.4.3


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("🔄 Loading merged Phi-3 model...")
MERGED_DIR = "/content/drive/MyDrive/workspace/orkspace/vakil-phi3-merged_model"

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

print("✅ Model loaded successfully!")


In [ ]:
def chat(instruction: str, context: str = "") -> str:
    prompt = f"<|user|>\n{context}\n{instruction}\n<|assistant|>"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=192,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    reply = text.split("<|assistant|>")[-1]
    for tok in ["<|user|>", "<|end|>", "<|eot_id|>", "<|endoftext|>"]:
        reply = reply.replace(tok, "")
    return reply.strip()

print(chat("Explain Section 302 IPC in simple words."))


In [ ]:
!pip uninstall -y torch transformers accelerate peft trl bitsandbytes
!pip install --upgrade --force-reinstall \
  torch==2.3.0 \
  transformers==4.42.4 \
  accelerate==0.30.1 \
  peft==0.11.1 \
  trl==0.9.6 \
  sentencepiece \
  huggingface_hub \
  tqdm \
  bitsandbytes


In [ ]:
!pip install langchain==0.2.11
!pip install langchain-community==0.2.10
!pip install langchain-huggingface==0.0.3
!pip install faiss-cpu


In [ ]:
# ========= IMPORTS =========
import faiss
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ========= LOAD EMBEDDINGS =========
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en")

# ========= LOAD FAISS INDEX =========
faiss_index = faiss.read_index(f"{FAISS_FOLDER}/index.faiss")

# ========= RECONSTRUCT VECTORSTORE =========
vectorstore = FAISS(
    index=faiss_index,
    embedding_function=embeddings.embed_query,
    docstore={},  # empty docstore, or you can rebuild from your source documents
    index_to_docstore_id={}  # empty mapping if original pickle is not loadable
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("🔥 FAISS vectorstore reconstructed (without pickle)!")


In [ ]:
!pip install --quiet --upgrade sentence-transformers


In [ ]:
# Upgrade Transformers and Accelerate
!pip install --upgrade --quiet transformers accelerate

# Install a version-compatible sentence-transformers
!pip install --upgrade --quiet "sentence-transformers>=2.2.2,<2.3.0"


In [ ]:
!pip install --quiet --upgrade transformers accelerate sentence-transformers
!pip install --quiet langchain_huggingface
!pip install --quiet faiss-cpu  # (ignore if already installed)


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en")
print("✅ Embeddings loaded successfully!")


In [ ]:
# ======================================================
# 🧠 RAG | FAISS + BGE-Small + Phi-3 (OOM FIXED)
# ======================================================

!pip install --quiet langchain-community langchain-core langchain
!pip install --quiet langchain_huggingface
!pip install --quiet faiss-cpu
!pip install --quiet transformers accelerate sentence-transformers

import torch, faiss, transformers
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ----- PATHS -----
FAISS_FOLDER = "/content/drive/MyDrive/rag_legal_project/faiss_index"
MODEL_PATH = "/content/drive/MyDrive/workspace/orkspace/vakil-phi3-merged_model"

print("⏳ Setting up embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")   # 🔥 NO OOM MODEL
print("✅ Embeddings ready")

print("⏳ Loading FAISS vector DB...")
vectorstore = FAISS.load_local(
    folder_path=FAISS_FOLDER,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ FAISS loaded & retriever ready")

print("⏳ Loading Phi-3 model...")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)
generator = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=450,
    temperature=0.1,
    top_p=0.9
)
print("🔥 Fine-tuned Phi-3 READY")

def rag_answer(question: str):
    docs = retriever.invoke(question)   # <--- fixed retriever method
    context = "\n\n".join(d.page_content for d in docs) if docs else ""
    if not context:
        return "📌 No relevant legal context found."

    prompt = f"""
You are a senior Indian legal expert. Use ONLY the retrieved context.
If context lacks answer, say:
"Insufficient legal basis in retrieved documents."

Context:
{context}

Question: {question}

Answer in detailed legal reasoning:
"""
    return generator(prompt)[0]["generated_text"]

QUESTION = "Explain Exception 4 to Section 300 IPC"
print("\n🟢 FINAL RAG RESPONSE:\n")
print(rag_answer(QUESTION))


In [ ]:
# ======================================================
# 🧠 RAG | FAISS + MiniLM + Phi-3 (8-BIT GPU – NO OOM)
# ======================================================

# ---- Install Dependencies ----
!pip install --quiet langchain-community langchain-core langchain
!pip install --quiet langchain_huggingface
!pip install --quiet faiss-cpu
!pip install --quiet accelerate
!pip install --quiet sentence-transformers
!pip install --quiet --upgrade transformers
!pip install --quiet bitsandbytes  # needed for 8-bit quantization

import torch, faiss, transformers
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForCausalLM

# ----- PATHS -----
FAISS_FOLDER = "/content/drive/MyDrive/rag_legal_project/faiss_index"
MODEL_PATH  = "/content/drive/MyDrive/workspace/orkspace/vakil-phi3-merged_model"

print("⌛ Setting up embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)
print("✅ Embeddings ready")

print("⌛ Loading FAISS DB...")
vectorstore = FAISS.load_local(
    folder_path=FAISS_FOLDER,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ FAISS loaded & retriever ready")

print("⌛ Loading Phi-3 model in 8-bit (GPU)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True,                # 🔥 QUANTIZATION (8-bit)
    trust_remote_code=True
)
print("🔥 Phi-3 READY — 8-bit GPU mode, NO OOM")

def generate_llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.0,
            top_p=0.9
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def rag_answer(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs) if docs else ""
    if not context:
        return "📌 No relevant legal context retrieved."

    prompt = f"""
You are a senior Indian legal expert. Use ONLY the retrieved context.
If the context lacks legal basis, reply:
"Insufficient legal basis in retrieved documents."

Retrieved Context:
{context}

Question: {question}

Detailed legal reasoning answer:
"""
    return generate_llm(prompt)

QUESTION = "Explain Exception 4 to Section 300 IPC"
print("\n🟢 FINAL RAG RESPONSE:\n")
print(rag_answer(QUESTION))


In [ ]:
import torch, transformers
print(torch.__version__)
print(transformers.__version__)


In [ ]:
!ls -lh "/content/drive/MyDrive/rag_legal_project/checkpoints" | head -100
!tail -n 20 "/content/drive/MyDrive/rag_legal_project/checkpoints/extract_errors.log" || echo "No extract_errors.log"


In [ ]:
# Quick list of recommended public dataset slugs for download
PUBLIC_KAGGLE_SLUGS = [
    "adarshsingh0903/legal-dataset-sc-judgments-india-19502024",
    "vangap/indian-supreme-court-judgments",
    "saurabhshahane/high-court-cases-in-india",
    "supratimhaldar/legal-case-pdf-dataset",
    "omkargurav/legal-documents"
]
# You can add your own dataset slugs or upload private datasets as well.
print("Recommended Kaggle dataset slugs (copy for download):", PUBLIC_KAGGLE_SLUGS)


In [ ]:
!ls -lh /content/legal_raw_data.csv


In [ ]:
##stage2-CLEANING AND PREPROCESSING

In [ ]:
#step1-clean and normalize raw text

In [ ]:
# Stage 2 - Step 1: Basic text cleaning & normalization
import pandas as pd
import re

# Load your raw CSV file
input_path = "/content/legal_raw_data.csv"
df = pd.read_csv(input_path)
print(f"✅ Loaded {len(df)} rows from {input_path}")

# Define a cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove multiple newlines
    text = re.sub(r'\n+', ' ', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # Remove page headers and footers (e.g., 'SUPREME COURT REPORTS [2011] 2 S.C.R.')
    text = re.sub(r'SUPREME COURT REPORTS.*?\d+ S\.C\.R\.', '', text, flags=re.IGNORECASE)
    # Remove common noisy tokens (page numbers, section marks, etc.)
    text = re.sub(r'Page\s*\d+|\[\d+\]|\(\d+\)|—', '', text)
    # Remove special characters except punctuation
    text = re.sub(r'[^A-Za-z0-9.,;:?!\'\"()\s-]', '', text)
    # Trim spaces
    return text.strip()

# Apply cleaning
df["clean_text"] = df["text"].apply(clean_text)

# Preview cleaned version
print("\n📄 Before cleaning:")
print(df["text"].iloc[0][:500])
print("\n✨ After cleaning:")
print(df["clean_text"].iloc[0][:500])

# Save cleaned data
output_path = "/content/legal_clean_data.csv"
df.to_csv(output_path, index=False)
print(f"\n💾 Cleaned dataset saved to: {output_path}")


In [ ]:
#🧭 Stage 2 – Step 2: Deduplication & Standardization

In [ ]:
# Stage 2 - Step 2: Remove duplicates and standardize text
import pandas as pd
import re

# Load cleaned data
input_path = "/content/legal_clean_data.csv"
df = pd.read_csv(input_path)
print(f"✅ Loaded {len(df)} rows from {input_path}")

# Remove duplicates based on text content
before = len(df)
df = df.drop_duplicates(subset=["clean_text"])
after = len(df)
print(f"🧹 Removed {before - after} duplicate pages. Remaining: {after}")

# Function to standardize text
def standardize_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase everything for uniformity
    text = text.lower()
    # Normalize spacing and punctuation
    text = re.sub(r'\s+', ' ', text)
    # Ensure proper sentence endings
    text = re.sub(r'\s*([?.!,;:])\s*', r'\1 ', text)
    return text.strip()

# Apply formatting
df["standard_text"] = df["clean_text"].apply(standardize_text)

# Quick preview
print("\n✨ Sample standardized text:\n")
print(df["standard_text"].iloc[0][:500])

# Save standardized file
output_path = "/content/legal_standard_data.csv"
df.to_csv(output_path, index=False)
print(f"\n💾 Standardized text saved to: {output_path}")


In [ ]:
#🧭 Stage 2 – Step 3: Metadata Mapping

In [ ]:
# Stage 2 - Step 3: Add metadata mapping for traceability
import pandas as pd
import hashlib

# Load standardized text
input_path = "/content/legal_standard_data.csv"
df = pd.read_csv(input_path)
print(f"✅ Loaded {len(df)} records from {input_path}")

# Ensure all required columns exist
expected_cols = ["source", "page_number", "standard_text"]
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    print(f"⚠️ Missing columns: {missing}")
else:
    print("✅ All required columns are present.")

# Generate unique document IDs based on filename
df["doc_id"] = df["source"].apply(lambda x: hashlib.md5(str(x).encode()).hexdigest()[:8])

# Create a combined metadata column for future retrieval
df["metadata"] = df.apply(lambda row: {
    "source": row["source"],
    "page_number": row["page_number"],
    "doc_id": row["doc_id"]
}, axis=1)

# Reorder columns
df = df[["doc_id", "source", "page_number", "standard_text", "metadata"]]

# Save final preprocessed file
output_path = "/content/legal_preprocessed_dataset.csv"
df.to_csv(output_path, index=False)
print(f"\n💾 Final preprocessed dataset saved to: {output_path}")

# Show preview
print("\n📊 Sample entry:\n")
print(df.head(2))


In [ ]:
# ⚖️ Stage 3 - chunking / splitting

In [ ]:
# Step 1: Decide and Test Chunk Size

In [ ]:
!pip install -U langchain langchain-community


In [ ]:
# Stage 3 - Step 1: Test chunk sizes using LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

# 1️⃣ Load preprocessed data
input_path = "/content/legal_preprocessed_dataset.csv"
df = pd.read_csv(input_path)
print(f"✅ Loaded {len(df)} rows from {input_path}")

# 2️⃣ Take a few samples of cleaned text
samples = df["standard_text"].dropna().sample(2, random_state=42).tolist()

# 3️⃣ Test different chunk sizes
for size in [400, 800, 1000]:
    print(f"\n{'='*40}\n🔹 Testing chunk_size={size} (with 100 overlap)")
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=100)

    for i, text in enumerate(samples):
        chunks = splitter.split_text(text)
        print(f"🧩 Sample {i+1}: {len(chunks)} chunks created.")
        print(f"First chunk (preview):\n{chunks[0][:400]}...\n")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

# 1️⃣ Load full preprocessed data
input_path = "/content/legal_preprocessed_dataset.csv"
df = pd.read_csv(input_path)
print(f"✅ Loaded {len(df)} rows.")

# 2️⃣ Initialize text splitter with recommended chunk size & overlap
chunk_size = 800
chunk_overlap = 150
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

# 3️⃣ Apply chunking to entire dataset
all_chunks = []
for idx, row in df.iterrows():
    text = str(row["standard_text"])  # Ensure text is string
    chunks = splitter.split_text(text)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "original_index": idx,
            "chunk_index": i,
            "chunk_text": chunk
        })

print(f"✅ Created {len(all_chunks)} total chunks from {len(df)} documents.")

# 4️⃣ Save chunked data
output_path = "/content/legal_chunked_dataset.csv"
chunked_df = pd.DataFrame(all_chunks)
chunked_df.to_csv(output_path, index=False)
print(f"✅ Saved chunked dataset to {output_path}")

In [ ]:
#⚖️ Stage 3 – Step 3: Embedding Chunks & Storing in Chroma


In [ ]:
!pip install -U langchain langchain-community chromadb openai


In [ ]:
!pip install -U langchain langchain-community chromadb sentence-transformers


In [ ]:
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1️⃣ Load chunked dataset
chunked_path = "/content/legal_chunked_dataset.csv"
df = pd.read_csv(chunked_path)
print(f"✅ Loaded {len(df)} chunks.")

# Ensure all chunk texts are strings
df["chunk_text"] = df["chunk_text"].astype(str)

# 2️⃣ Initialize free HuggingFace embeddings
# Model "sentence-transformers/all-MiniLM-L6-v2" is small, fast, free
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3️⃣ Convert chunks into list of texts
documents = df["chunk_text"].tolist()

# 4️⃣ Create / persist Chroma vector database
persist_directory = "/content/chroma_legal_db"
vectordb = Chroma.from_texts(
    texts=documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

# 5️⃣ Persist to disk
vectordb.persist()
print(f"✅ Vector database saved to {persist_directory}")

In [ ]:
#Stage 4: Setup Retrieval & Query Pipeline

In [ ]:
# Step 1: Initialize a Retriever from Chroma

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Step 1: Load your Chroma vector database
persist_directory = "/content/chroma_legal_db"

# Use a free embedding model from HuggingFace
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

# Step 2: Create a retriever
retriever = vectordb.as_retriever(
    search_type="similarity",   # Fetch top relevant chunks
    search_kwargs={"k": 5}      # Top 5 chunks per query
)

# Test the retriever with a sample query
query = "What is the procedure for filing a civil case?"
results = retriever.invoke(query)

# Print retrieved chunk summaries
for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content[:500])  # print first 500 chars for preview

In [ ]:
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Important
#Step 2: Connect Retriever to a Free LLM

In [ ]:
!pip install --upgrade langchain langchain_community
!pip install gpt4all


In [ ]:
!pip install --upgrade langchain langchain-community gpt4all


In [ ]:
!pip install --upgrade langchain langchain-community transformers accelerate sentence-transformers


In [ ]:
# Core imports after installing/upgrading
# The imports below are causing ModuleNotFoundError errors,
# likely due to an issue with the LangChain installation or environment.
# You need to fix your LangChain installation for this cell to work.
# Specifically, check installations of langchain and langchain-community.
# from langchain.llms import HuggingFacePipeline # This import is failing
# from langchain.chains import RetrievalQA # This import is also failing
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
!pip install -qU langchain-chroma


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Example path to access files in your Drive
!ls -lh "/content/drive/MyDrive/LegalData"


In [ ]:
!ls -lh /content


# Task
Modify cell `Pr-CUebW_Cb2` to load the Phi-3 model using `BitsAndBytesConfig` for 4-bit quantization to resolve the OutOfMemoryError and then execute the modified cell to load the model and test the RAG pipeline.

## Modify cell to use 4-bit quantization

### Subtask:
Modify the cell to load the Phi-3 model using `BitsAndBytesConfig` for 4-bit quantization to resolve the OutOfMemoryError.


**Reasoning**:
The subtask requires modifying an existing cell to implement 4-bit quantization for the Phi-3 model. I will update the code in `Pr-CUebW_Cb2` to import `BitsAndBytesConfig`, define the quantization configuration, and pass it to the model loading function, along with ensuring the `low_cpu_mem_usage` parameter is correctly set.



In [ ]:
# ======================================================
# ➗ RAG | FAISS + MiniLM + Phi-3 (OOM-PROOF CPU MODE)
# ======================================================

!pip install --quiet langchain-community langchain-core langchain
!pip install --quiet langchain_huggingface
!pip install --quiet faiss-cpu
!pip install --quiet transformers accelerate sentence-transformers bitsandbytes

import torch, faiss, transformers
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import BitsAndBytesConfig # Import BitsAndBytesConfig

# ----- PATHS -----
FAISS_FOLDER = "/content/drive/MyDrive/rag_legal_project/faiss_index"
MODEL_PATH = "/content/drive/MyDrive/workspace/orkspace/vakil-phi3-merged_model"

print("⌛ Setting up embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}   # ← force CPU for embeddings
)
print("✅ Embeddings ready")

print("⌛ Loading FAISS DB...")
vectorstore = FAISS.load_local(
    folder_path=FAISS_FOLDER,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ FAISS loaded & retriever ready")

print("⌛ Loading Phi-3 model (4-bit quantized)...")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Define 4-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    # torch_dtype=torch.float32, # No longer needed, as bnb_4bit_compute_dtype handles this
    device_map="auto",             # ← Use "auto" for quantization
    trust_remote_code=True,
    low_cpu_mem_usage=True,        # ← reduces RAM usage
    quantization_config=quantization_config # ← Pass the quantization config
)
print("🔥 Fine-tuned Phi-3 READY (4-bit Quantized)")

def generate_llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.1,
            top_p=0.9
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

def rag_answer(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs) if docs else ""
    if not context:
        return "┬ No relevant legal context retrieved."

    prompt = f"""
You are a senior Indian legal expert. Use ONLY the retrieved context.
If no answer exists in context, reply:
"Insufficient legal basis in retrieved documents."

Context:
{context}

Question: {question}

Give a detailed legal interpretation:
"""
    return generate_llm(prompt)

QUESTION = "Explain Exception 4 to Section 300 IPC"
print("\n● FINAL RAG RESPONSE:\n")
print(rag_answer(QUESTION))

**Reasoning**:
The `AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'` arises from a compatibility issue with the `transformers` library's caching mechanism during generation. Disabling `use_cache` in the `model.generate` method can bypass this specific error.



In [ ]:
# ======================================================
# ➗ RAG | FAISS + MiniLM + Phi-3 (OOM-PROOF CPU MODE)
# ======================================================

!pip install --quiet langchain-community langchain-core langchain
!pip install --quiet langchain_huggingface
!pip install --quiet faiss-cpu
!pip install --quiet transformers accelerate sentence-transformers bitsandbytes

import torch, faiss, transformers
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import BitsAndBytesConfig # Import BitsAndBytesConfig

# ----- PATHS -----
FAISS_FOLDER = "/content/drive/MyDrive/rag_legal_project/faiss_index"
MODEL_PATH = "/content/drive/MyDrive/workspace/orkspace/vakil-phi3-merged_model"

print("⌛ Setting up embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}   # ← force CPU for embeddings
)
print("✅ Embeddings ready")

print("⌛ Loading FAISS DB...")
vectorstore = FAISS.load_local(
    folder_path=FAISS_FOLDER,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ FAISS loaded & retriever ready")

print("⌛ Loading Phi-3 model (4-bit quantized)...")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Define 4-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",             # ← Use "auto" for quantization
    trust_remote_code=True,
    low_cpu_mem_usage=True,        # ← reduces RAM usage
    quantization_config=quantization_config # ← Pass the quantization config
)
print("🔥 Fine-tuned Phi-3 READY (4-bit Quantized)")

def generate_llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.1,
            top_p=0.9,
            use_cache=False # Explicitly disable cache to avoid 'seen_tokens' error
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

def rag_answer(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs) if docs else ""
    if not context:
        return "┬ No relevant legal context retrieved."

    prompt = f"""
You are a senior Indian legal expert. Use ONLY the retrieved context.
If no answer exists in context, reply:
"Insufficient legal basis in retrieved documents."

Context:
{context}

Question: {question}

Give a detailed legal interpretation:
"""
    return generate_llm(prompt)

QUESTION = "Explain Exception 4 to Section 300 IPC"
print("\n● FINAL RAG RESPONSE:\n")
print(rag_answer(QUESTION))


**Reasoning**:
The `AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'` is a known issue when `use_cache=True` with certain `transformers` versions and models. Disabling caching during generation (`use_cache=False`) is a common workaround to resolve this specific error, which was not done in the previous execution.



In [ ]:
# ======================================================
# ➗ RAG | FAISS + MiniLM + Phi-3 (OOM-PROOF CPU MODE)
# ======================================================

!pip install --quiet langchain-community langchain-core langchain
!pip install --quiet langchain_huggingface
!pip install --quiet faiss-cpu
!pip install --quiet transformers accelerate sentence-transformers bitsandbytes

import torch, faiss, transformers
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import BitsAndBytesConfig # Import BitsAndBytesConfig

# ----- PATHS -----
FAISS_FOLDER = "/content/drive/MyDrive/rag_legal_project/faiss_index"
MODEL_PATH = "/content/drive/MyDrive/workspace/orkspace/vakil-phi3-merged_model"

print("⌛ Setting up embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}   # ← force CPU for embeddings
)
print("✅ Embeddings ready")

print("⌛ Loading FAISS DB...")
vectorstore = FAISS.load_local(
    folder_path=FAISS_FOLDER,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ FAISS loaded & retriever ready")

print("⌛ Loading Phi-3 model (fully on CPU)...")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Define 4-bit quantization configuration (not used for full CPU offloading, but kept if user changes mind)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,     # ← Use float32 for CPU
    device_map=None,               # ← Explicitly set device_map to None to force CPU offloading
    trust_remote_code=True,
    # quantization_config=quantization_config # ← Removed for full CPU loading
    low_cpu_mem_usage=True         # Keep low_cpu_mem_usage for potentially larger models on CPU
)
print("🔥 Fine-tuned Phi-3 READY (fully on CPU)")

def generate_llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu") # ← Ensure inputs are on CPU
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.1,
            top_p=0.9,
            use_cache=False # Explicitly disable cache to avoid 'seen_tokens' error
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

def rag_answer(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs) if docs else ""
    if not context:
        return "┡ No relevant legal context retrieved."

    prompt = f"""
You are a senior Indian legal expert. Use ONLY the retrieved context.
If no answer exists in context, reply:
"Insufficient legal basis in retrieved documents."

Context:
{context}

Question: {question}

Give a detailed legal interpretation:
"""
    return generate_llm(prompt)

QUESTION = "Explain Exception 4 to Section 300 IPC"
print("\n● FINAL RAG RESPONSE:\n")
print(rag_answer(QUESTION))


## Analyze memory usage

### Subtask:
Analyze the current GPU memory usage to identify potential areas for further optimization, such as checking if any other processes are consuming GPU memory.


**Reasoning**:
To analyze the current GPU memory usage and identify any processes consuming GPU memory, I will execute the `!nvidia-smi` command as instructed.



In [ ]:
!nvidia-smi

# Task
Next, I will execute the code in cell `ee13b56b` to try generating with a very small `max_new_tokens` (set to 20). This will help determine if the length of the generated output is the primary cause of the `OutOfMemoryError`, even after 4-bit quantization and disabling `use_cache`. This addresses the second point of the plan.

## Adjust `max_new_tokens` further

### Subtask:
Reduce the `max_new_tokens` parameter in the model generation to an even smaller value (e.g., 20) in cell `ee13b56b` to determine if output length is the bottleneck for the OutOfMemoryError.


**Reasoning**:
To address the OutOfMemoryError, I will modify the `max_new_tokens` parameter in the `model.generate` call within the `generate_llm` function in cell `ee13b56b` from 250 to 20. This will reduce the length of the generated output, potentially mitigating memory issues. The cell is `ee13b56b`.



In [ ]:
# ======================================================
# ─ RAG | FAISS + MiniLM + Phi-3 (OOM-PROOF CPU MODE)
# ======================================================

!pip install --quiet langchain-community langchain-core langchain
!pip install --quiet langchain_huggingface
!pip install --quiet faiss-cpu
!pip install --quiet transformers accelerate sentence-transformers bitsandbytes

import torch, faiss, transformers
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import BitsAndBytesConfig # Import BitsAndBytesConfig

# ----- PATHS -----
FAISS_FOLDER = "/content/drive/MyDrive/rag_legal_project/faiss_index"
MODEL_PATH = "/content/drive/MyDrive/workspace/orkspace/vakil-phi3-merged_model"

print("⌛ Setting up embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}   # ← force CPU for embeddings
)
print("✅ Embeddings ready")

print("⌛ Loading FAISS DB...")
vectorstore = FAISS.load_local(
    folder_path=FAISS_FOLDER,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ FAISS loaded & retriever ready")

print("⌛ Loading Phi-3 model (4-bit quantized)...")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Define 4-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",             # ← Use "auto" for quantization
    trust_remote_code=True,
    low_cpu_mem_usage=True,        # ← reduces RAM usage
    quantization_config=quantization_config # ← Pass the quantization config
)
print("🔥 Fine-tuned Phi-3 READY (4-bit Quantized)")

def generate_llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=20,  # Reduced to 20 tokens
            temperature=0.1,
            top_p=0.9,
            use_cache=False # Explicitly disable cache to avoid 'seen_tokens' error
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

def rag_answer(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs) if docs else ""
    if not context:
        return "┬ No relevant legal context retrieved."

    prompt = f"""
You are a senior Indian legal expert. Use ONLY the retrieved context.
If no answer exists in context, reply:
"Insufficient legal basis in retrieved documents."

Context:
{context}

Question: {question}

Give a detailed legal interpretation:
"""
    return generate_llm(prompt)

QUESTION = "Explain Exception 4 to Section 300 IPC"
print("\n● FINAL RAG RESPONSE:\n")
print(rag_answer(QUESTION))

**Reasoning**:
The previous execution of cell `ee13b56b` with `max_new_tokens=20` still resulted in an `AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'`, indicating that reducing the output length alone did not resolve the generation issue. This suggests the bottleneck might not solely be the output length. The next step as per the main task is to analyze GPU memory usage.



In [ ]:
!nvidia-smi